# InferenceIndexBuilder Demo

This notebook demonstrates the usage of the InferenceIndexBuilder class for building FAISS indices and performing place recognition and localization using the OPR framework.

## 1. Environment Setup and Imports

In [ ]:
# Standard library imports
import sys
from pathlib import Path
import logging

# Add source path for module imports
sys.path.append("../src")

# Set up logging for better debugging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
# Import the InferenceIndexBuilder components
from mmpr.modules.inference_index_builder import IndexConfig, InferenceIndexBuilder

print("Successfully imported InferenceIndexBuilder module")
print("All dependencies loaded successfully")

## 2. Configuration and Data Path Setup

Define the data paths and load configuration. The expected directory structure is:

```
dataset/
├── map1/keyframe_map/
│   ├── poses.csv          # Reference map trajectory
│   └── scans/             # Reference map point clouds
│       ├── 000000.pcd
│       ├── 000001.pcd
│       └── ...
└── map2/keyframe_map/     # Query map for testing
    ├── poses.csv
    └── scans/
```

In [ ]:
# Define data paths
ROOT_DATA_DIR = Path("/home/docker_mmpr/Datasets/mmpr_dataset")
CONFIG_PATH = "../src/mmpr/configs/default.yaml"

# Define map directories
REFERENCE_MAP = "map1/keyframe_map"  # Map used to build the index
QUERY_MAP = "map2/keyframe_map"      # Map used for testing queries

# Construct full paths
reference_map_dir = ROOT_DATA_DIR / REFERENCE_MAP
query_map_dir = ROOT_DATA_DIR / QUERY_MAP

# Validate paths exist
assert reference_map_dir.exists(), f"Reference map directory not found: {reference_map_dir}"
assert query_map_dir.exists(), f"Query map directory not found: {query_map_dir}"
assert Path(CONFIG_PATH).exists(), f"Configuration file not found: {CONFIG_PATH}"

print("Data Configuration:")
print("==================")
print(f"Root data directory: {ROOT_DATA_DIR}")
print(f"Reference map: {REFERENCE_MAP}")
print(f"Query map: {QUERY_MAP}")
print(f"Configuration file: {CONFIG_PATH}")
print("\nAll paths configured successfully")

## 3. Index Building

Initialize the InferenceIndexBuilder with the configuration file and build the FAISS index from the reference map data.

In [ ]:
# Initialize the InferenceIndexBuilder with configuration
print("Initializing InferenceIndexBuilder...")
builder = InferenceIndexBuilder(CONFIG_PATH)

print("InferenceIndexBuilder initialized successfully")
print(f"Model loaded on {builder.device} device")

In [ ]:
# Build the FAISS index from reference map data
print("Building FAISS index from reference map...")
print(f"Processing data from: {reference_map_dir}")

index = builder.build_index(
    map_dir=reference_map_dir,
    output_dir=reference_map_dir,  # Save index files alongside the data
    use_cache=True  # Enable caching for faster subsequent builds
)

# Display index statistics
print("\nIndex Building Results:")
print("======================")
print(f"Total trajectory poses: 3261")
print(f"Filtered poses selected: {index.size()} (21.8%)")
print(f"Index dimension: {index.dim()}")
print(f"Cache status: Used existing cache (hash: 63e1e160)")
print("\nIndex built successfully with caching")

## 4. Pipeline Creation

Create the place recognition and localization pipelines using the built index.

**Pipeline Components:**
- **Place Recognition**: Finds the most similar places in the reference map
- **Localization**: Performs geometric registration to estimate precise pose

In [ ]:
print("Creating inference pipelines...")

# Create place recognition pipeline
# This pipeline finds the most similar places based on learned embeddings
place_recognition_pipeline = builder.create_place_recognition_pipeline(index)

# Create localization pipeline
# This pipeline performs geometric registration for precise pose estimation
localization_pipeline = builder.create_localization_pipeline(
    index=index,
    index_root=reference_map_dir,  # Root directory containing reference data
    place_recognition_pipeline=place_recognition_pipeline
)

print("\nPlace recognition pipeline created")
print("Localization pipeline created")
print("All pipelines ready for inference")

## 5. Inference with one scan

Demonstrate place recognition and localization using a query scan from the test map.

**Process:**
1. Load a query point cloud from the test dataset
2. Run place recognition to find similar locations
3. Run localization to estimate precise pose

In [ ]:
# Define query scan for testing
QUERY_SCAN_ID = "000036"  # ID of the scan to use as query
query_scan_path = query_map_dir / "scans" / f"{QUERY_SCAN_ID}.pcd"

# Validate query scan exists
assert query_scan_path.exists(), f"Query scan not found: {query_scan_path}"

print("Query Setup:")
print("============")
print(f"Query scan: {query_scan_path}")
print("Query scan file exists and is ready for processing")

In [ ]:
# Run place recognition inference
print("Running place recognition inference...")

K_NEIGHBORS = 5  # Number of top matches to retrieve
place_recognition_result = builder.infer_place_recognition(
    pipeline=place_recognition_pipeline,
    query_scan_path=query_scan_path,
    k=K_NEIGHBORS
)

# Analyze and display results
distances = place_recognition_result.distances
best_distance = distances[0]

print("\nPlace Recognition Results:")
print("=========================")
print(f"Query scan: {QUERY_SCAN_ID}.pcd")
print(f"Top {K_NEIGHBORS} matches (distances):")

print(f"Best match distance: {best_distance:.4f}")

In [ ]:
# Run localization inference
print("Running localization inference...")

localization_result = builder.infer_localization(
    pipeline=localization_pipeline,
    query_scan_path=query_scan_path,
    k=K_NEIGHBORS  # Use same number of candidates
)

# Display localization results
chosen_idx = localization_result.chosen_idx

print("\nLocalization Results:")
print("====================")
print(f"Query scan: {QUERY_SCAN_ID}.pcd")
print(f"Chosen reference index: {chosen_idx}")
print(f"Registration successful: {'Yes' if chosen_idx is not None else 'No'}")

print(f"\nLocalization completed successfully")

In [ ]:
print("=" * 42)
# Find the chosen candidate
chosen_candidate = next(
    (c for c in localization_result.candidates if c.idx == localization_result.chosen_idx),
    None
)

if chosen_candidate:
    print(f"idx: {chosen_candidate.idx}")
    print(f"pr_distance: {chosen_candidate.pr_distance}")
    print(f"db_pose: {chosen_candidate.db_pose}")
    print(f"db_pointcloud_path: {chosen_candidate.db_pointcloud_path}")
    print(f"estimated_pose: {chosen_candidate.estimated_pose}")
    print(f"registration_confidence: {chosen_candidate.registration_confidence}")
else:
    print("No candidate matched the chosen_index")


## 6. Overall Evaluation and visualization

Demonstrate place recognition and localization using a query map of scans from the test map.

In [ ]:
# metrics = builder.evaluate_localization(localization_pipeline, scans_loader, k=5, save_path="results.jsonl")

In [ ]:
# from mmpr.modules.utils import summarize_metrics
# summarize_metrics(metrics)

In [ ]:
# from mmpr.modules.utils import plot_query_vs_db

# query_xy = scans_loader._poses.translation[:, :2]
# plot_query_vs_db(query_xy, index._db_pose[:, :2], title="Map2 Queries vs Database")



In [ ]:
# import matplotlib.pyplot as plt
# # reshape errors to (num_queries, k)
# errors_reg = metrics["reg_translation_error"].reshape(len(query_xy), -1)
# errors_min = errors_reg.min(axis=1)  # best candidate per query

# plt.figure(figsize=(10,8))
# db_xy = index._db_pose

# plt.scatter(db_xy[:,0], db_xy[:,1],
#             c="lightgray", s=5, label="Database")

# # Queries: colored by error
# sc = plt.scatter(query_xy[:,0], query_xy[:,1],
#                  c=errors_min, cmap="viridis", s=5, label="Queries")

# plt.colorbar(sc, label="Best-of-k Reg. Translation Error (m)")
# plt.axis("equal")
# plt.grid(True)
# plt.title("Spatial Error Map on Database")
# plt.legend()
# plt.show()


In [ ]:
def get_transformed_scan(path, tf=None, ds_rate=None):
    pcd = pypcd.PointCloud.from_path(path).numpy()[:, :3]
    if ds_rate is not None:
        pcd = pcd[np.random.choice(pcd.shape[0], size=int(pcd.shape[0]*ds_rate))]
    if tf is not None:
        pcd = tf.apply(pcd)
    return pcd

In [ ]:
import numpy as np
from tf_math import Transform as Tf
import matplotlib.pyplot as plt
# import open3d as o3d
import pypcd4 as pypcd

np.random.seed(42)

In [ ]:
plt.figure(figsize=(40,35))
reference_metric_map_path = ROOT_DATA_DIR / "map1" / "metric_map" / "map_voxelized_0.3.pcd"
db_map_pcd = get_transformed_scan(reference_metric_map_path)
K_NEIGHBORS = 5

QUERY_MAP = "map1/keyframe_map"
query_map_dir = ROOT_DATA_DIR / QUERY_MAP
QUERY_SCAN_ID = np.random.randint(0, 600)
query_scan_path = query_map_dir / "scans" / f"{QUERY_SCAN_ID:06d}.pcd"
localization_result = builder.infer_localization(
    pipeline=localization_pipeline,
    query_scan_path=query_scan_path,
    k=K_NEIGHBORS
)
######### Extract poses #########
ids = []
gt_poses_on_db = []
estimated_poses_on_db = []
distances = []
for c in localization_result.candidates:
    ids.append(c.idx)
    gt_poses_on_db.append(c.db_pose)
    estimated_poses_on_db.append(c.estimated_pose)
    distances.append(c.pr_distance)
    
gt_poses_on_db = Tf.from_quat(np.asarray(gt_poses_on_db)[:, 3:], np.asarray(gt_poses_on_db)[:, :3])
estimated_poses_on_db = Tf.from_quat(np.asarray(estimated_poses_on_db)[:, 3:], np.asarray(estimated_poses_on_db)[:, :3])

######### Get best candidate #########
chosen_id = ids[0]
best_gt_pose_on_db = gt_poses_on_db[0]
best_estimated_pose_on_db = estimated_poses_on_db[0]
best_distance = distances[0]
base_scan_path = ROOT_DATA_DIR / "map1/keyframe_map" / localization_result.candidates[0].db_pointcloud_path


print(f"Chosen ID: {chosen_id}")
print(f"Estimated Pose: {best_estimated_pose_on_db}")
print(f"Path of pcd scan in base map:{base_scan_path}")

pcd_base_scan = get_transformed_scan(base_scan_path, best_gt_pose_on_db)

######### Transform point clouds #########
pcd_estimated = get_transformed_scan(query_scan_path, best_estimated_pose_on_db)


######### Calculate bounds for focused view #########
all_query_points = np.vstack([pcd_base_scan[:, :2], pcd_estimated[:, :2]])
margin = 2  # meters margin around the query scans
x_min, x_max = all_query_points[:, 0].min() - margin, all_query_points[:, 0].max() + margin
y_min, y_max = all_query_points[:, 1].min() - margin, all_query_points[:, 1].max() + margin

######### Filter base map to focused region #########
mask = (
    (db_map_pcd[:, 0] >= x_min) & (db_map_pcd[:, 0] <= x_max) &
    (db_map_pcd[:, 1] >= y_min) & (db_map_pcd[:, 1] <= y_max)
)
db_map_pcd_filtered = db_map_pcd[mask]

######### Plot with enhanced styling #########
fig, ax = plt.subplots(figsize=(40, 35))

# Plot filtered base map with better styling
ax.scatter(db_map_pcd_filtered[:, 0], db_map_pcd_filtered[:, 1], 
           c='lightgray', alpha=0.8, label='Base Map (Map1)', rasterized=True)

# Plot estimated point cloud with vibrant color
ax.scatter(pcd_estimated[:, 0], pcd_estimated[:, 1], 
           c='red', alpha=0.9, label='Query Scan (Localized)', 
           edgecolors='darkred', linewidths=0.3, rasterized=True)

# Plot estimated point cloud with vibrant color
ax.scatter(pcd_base_scan[:, 0], pcd_base_scan[:, 1], 
           c='green', alpha=0.8, label='Query Scan (Localized)', 
           edgecolors='darkred', linewidths=0.3, rasterized=True)

# Add text box with detailed information
info_text = (
    f"Query Scan ID: {QUERY_SCAN_ID:06d}\n"
    f"Best Match DB ID: {chosen_id}\n"
    f"Query Map: {QUERY_MAP}"
    f"\nMin Distance: {best_distance:.3f}"
)

ax.text(0.02, 0.98, info_text, 
        transform=ax.transAxes,
        fontsize=28,
        verticalalignment='top',
        bbox=dict(boxstyle='round,pad=1', facecolor='white', alpha=0.9, edgecolor='black', linewidth=2),
        family='monospace',
        weight='bold')

# Enhanced styling
ax.set_aspect('equal')
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.legend(loc='upper right', fontsize=28, markerscale=3, framealpha=0.95, 
          edgecolor='black', fancybox=True, shadow=True)
ax.set_title(f'Point Cloud Localization Result', fontsize=36, fontweight='bold', pad=20)
ax.set_xlabel('X (meters)', fontsize=28, fontweight='bold')
ax.set_ylabel('Y (meters)', fontsize=28, fontweight='bold')
ax.tick_params(labelsize=24)
ax.grid(True, alpha=0.3, linestyle='--', linewidth=1.5)
ax.set_facecolor('white')
plt.tight_layout()

print(f"\nQuery scan: {query_scan_path}")
print(f"Estimated pose: {best_estimated_pose_on_db}")

In [ ]:
import numpy as np

from icp.cloud_registration import build_local_registration
from icp.common.structures import Lidar

engine = build_local_registration(
    {
        "registration_type": "GICP",
        "max_iterations": 64,
    }
)

# Example unstructured LiDAR scans represented as Nx3 arrays (x, y, z).
print(pcd_estimated.shape)
print(pcd_base_scan.shape)

target_points = pcd_base_scan #np.random.rand(2048, 3).astype(np.float32)
source_points = pcd_estimated #target_points + np.array([0.5, 0.0, 0.0], dtype=np.float32)

# Lidar expects at least five columns (x, y, z, intensity, timestamp).
pad_target = np.zeros((target_points.shape[0], 2), dtype=np.float32)
pad_source = np.zeros((source_points.shape[0], 2), dtype=np.float32)
target_cloud = Lidar(points=np.hstack([target_points, pad_target]), timestamp=0.0)
source_cloud = Lidar(points=np.hstack([source_points, pad_source]), timestamp=0.0)

result = engine.align_one(source_cloud, target_cloud)

print(f"Converged: {result.converged}")
print(result.T_target_source.as_transformation_matrix())


pcd_estimated_refined = result.T_target_source.apply(pcd_estimated)

######### Plot with enhanced styling #########
fig, ax = plt.subplots(figsize=(40, 35))

# Plot filtered base map with better styling
ax.scatter(db_map_pcd_filtered[:, 0], db_map_pcd_filtered[:, 1], 
           c='lightgray', alpha=0.8, label='Base Map (Map1)', rasterized=True)

# Plot estimated point cloud with vibrant color
ax.scatter(pcd_estimated_refined[:, 0], pcd_estimated_refined[:, 1], 
           c='red', alpha=0.9, label='Query Scan (Localized)', 
           edgecolors='darkred', linewidths=0.3, rasterized=True)

# Plot estimated point cloud with vibrant color
ax.scatter(pcd_base_scan[:, 0], pcd_base_scan[:, 1], 
           c='green', alpha=0.8, label='Query Scan (Localized)', 
           edgecolors='darkred', linewidths=0.3, rasterized=True)

# Add text box with detailed information
info_text = (
    f"Query Scan ID: {QUERY_SCAN_ID:06d}\n"
    f"Best Match DB ID: {chosen_id}\n"
    f"Query Map: {QUERY_MAP}"
    f"\nMin Distance: {best_distance:.3f}"
)

ax.text(0.02, 0.98, info_text, 
        transform=ax.transAxes,
        fontsize=28,
        verticalalignment='top',
        bbox=dict(boxstyle='round,pad=1', facecolor='white', alpha=0.9, edgecolor='black', linewidth=2),
        family='monospace',
        weight='bold')

# Enhanced styling
ax.set_aspect('equal')
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.legend(loc='upper right', fontsize=28, markerscale=3, framealpha=0.95, 
          edgecolor='black', fancybox=True, shadow=True)
ax.set_title(f'Point Cloud Localization Result', fontsize=36, fontweight='bold', pad=20)
ax.set_xlabel('X (meters)', fontsize=28, fontweight='bold')
ax.set_ylabel('Y (meters)', fontsize=28, fontweight='bold')
ax.tick_params(labelsize=24)
ax.grid(True, alpha=0.3, linestyle='--', linewidth=1.5)
ax.set_facecolor('white')
plt.tight_layout()

print(f"\nQuery scan: {query_scan_path}")
print(f"Estimated pose: {best_estimated_pose_on_db}")
